In [ ]:
!nvidia-smi

Fri Dec 19 03:26:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q \
    pandas \
    spacy \
    underthesea \
    transformers \
    sentencepiece \
    awesome-align

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import json
import unicodedata
import pandas as pd
from tqdm import tqdm

In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
from transformers import pipeline

MAX_LENGTH = 512

vi_spell_corrector = pipeline(
    "text2text-generation",
    model="bmd1905/vietnamese-correction-v2",
    device=0  # GPU T4
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/961 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
def spell_check_vi(sentences, batch_size=8):
    corrected = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        preds = vi_spell_corrector(batch, max_new_tokens=MAX_LENGTH)
        corrected.extend([p["generated_text"] for p in preds])
    return corrected

In [ ]:
# Upload file manually or mount Google Drive
df = pd.read_csv("data.csv")

df["English"] = df["English"].apply(normalize_text)
df["Vietnamese"] = df["Vietnamese"].apply(normalize_text)

df = df[(df["English"] != "") & (df["Vietnamese"] != "")]
df.head()

,English,Vietnamese
0,Do it.,Làm đi
1,Today is your glory-,Hôm nay là nguy huy hoàng.
2,All my prep was in there.,Tất cả tài liệu của tôi đều ở trong đó .
3,He learned that a ranch ain't only beef but it...,Ổng biết được là một nông trại không chỉ có bò...
4,"- Just about, sir.","- Thưa Ngài, vừa xong."


In [ ]:
df["Vietnamese_corrected"] = spell_check_vi(df["Vietnamese"].tolist())
df[["Vietnamese", "Vietnamese_corrected"]].head()

100%|██████████| 1500/1500 [36:16<00:00,  1.45s/it]


,Vietnamese,Vietnamese_corrected
0,Làm đi,Làm đi.
1,Hôm nay là nguy huy hoàng.,Hôm nay là nguy huy hoàng.
2,Tất cả tài liệu của tôi đều ở trong đó .,Tất cả tài liệu của tôi đều ở trong đó .
3,Ổng biết được là một nông trại không chỉ có bò...,Ổng biết được là một nông trại không chỉ có bò...
4,"- Thưa Ngài, vừa xong.","- Thưa Ngài, vừa xong."


In [ ]:
def length_ratio(en, vi):
    return len(en.split()) / max(len(vi.split()), 1)

df = df[df.apply(
    lambda r: 0.5 < length_ratio(r["English"], r["Vietnamese_corrected"]) < 2.0,
    axis=1
)]

print("Remaining sentence pairs:", len(df))

Remaining sentence pairs: 10337


In [ ]:
import spacy
nlp_en = spacy.load("en_core_web_sm")

def tokenize_en(sent):
    return [tok.text for tok in nlp_en(sent)]

In [ ]:
from underthesea import word_tokenize

def tokenize_vi(sent):
    return word_tokenize(sent, format="text").split()

In [ ]:
df["en_tokens"] = df["English"].apply(tokenize_en)
df["vi_tokens"] = df["Vietnamese"].apply(tokenize_vi)

df[["en_tokens", "vi_tokens"]].head()

,en_tokens,vi_tokens
0,"[Do, it, .]","[Làm, đi]"
1,"[Today, is, your, glory-]","[Hôm_nay, là, nguy_huy_hoàng, .]"
2,"[All, my, prep, was, in, there, .]","[Tất_cả, tài_liệu, của, tôi, đều, ở, trong, đó..."
3,"[He, learned, that, a, ranch, ai, n't, only, b...","[Ổng, biết, được, là, một, nông_trại, không_ch..."
4,"[-, Just, about, ,, sir, .]","[-, Thưa, Ngài, ,, vừa, xong, .]"


In [ ]:
with open("parallel.txt", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(" ".join(row["en_tokens"]))
        f.write(" ||| ")
        f.write(" ".join(row["vi_tokens"]))
        f.write("\n")

In [ ]:
!awesome-align \
  --model_name_or_path bert-base-multilingual-cased \
  --data_file parallel.txt \
  --output_file align.txt \
  --extraction softmax

2025-12-19 04:08:37.629813: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766117317.664115   22748 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766117317.673665   22748 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766117317.698433   22748 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766117317.698473   22748 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766117317.698482   22748 computation_placer.cc:177] computation placer alr

In [ ]:
def parse_alignment(line):
    return [tuple(map(int, p.split("-"))) for p in line.strip().split()]

In [ ]:
aligned_data = []

with open("align.txt", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        aligned_data.append({
            "en": {
                "tokens": df.iloc[idx]["en_tokens"]
            },
            "vi": {
                "tokens": df.iloc[idx]["vi_tokens"]
            },
            "alignment": parse_alignment(line)
        })

with open("aligned_en_vi.json", "w", encoding="utf-8") as f:
    json.dump(aligned_data, f, ensure_ascii=False, indent=2)

print("Saved to aligned_en_vi.json")

Saved to aligned_en_vi.json


In [ ]:
!pip install nltk

In [ ]:
import json
from collections import defaultdict, Counter
from tqdm import tqdm

import nltk
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
nltk.download('averaged_perceptron_tagger_eng')
nltk.download("omw-1.4")
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords
from nltk.wsd import lesk
from underthesea import pos_tag as vi_pos_tag

STOPWORDS = set(stopwords.words("english"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
DATA_PATH = "/content/aligned_en_vi.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

len(corpus)

12000

In [ ]:
def penn_to_wn_pos(tag):
    if tag.startswith("N"):
        return wn.NOUN
    if tag.startswith("V"):
        return wn.VERB
    if tag.startswith("J"):
        return wn.ADJ
    if tag.startswith("R"):
        return wn.ADV
    return None

In [ ]:
lemmatizer = WordNetLemmatizer()

en_lemma_pos_counter = Counter()

for sent in corpus:
    tokens = sent["en"]["tokens"]
    tagged = pos_tag(tokens)

    for word, tag in tagged:
        wn_pos = penn_to_wn_pos(tag)
        if wn_pos is None:
            continue

        word = word.lower()
        lemma = lemmatizer.lemmatize(word, wn_pos)

        # chỉ giữ lemma có WordNet synset thật
        if wn.synsets(lemma, pos=wn_pos):
            en_lemma_pos_counter[(lemma, wn_pos)] += 1

len(en_lemma_pos_counter)

6155

In [ ]:
MIN_FREQ = 3

valid_en_lemmas = {
    (lemma, pos)
    for (lemma, pos), cnt in en_lemma_pos_counter.items()
    if cnt >= MIN_FREQ
}

len(valid_en_lemmas)

1869

In [ ]:
POS_STR = {
    wn.NOUN: "n",
    wn.VERB: "v",
    wn.ADJ: "a",
    wn.ADV: "r"
}

en_sense_inventory = {}

for lemma, wn_pos in tqdm(valid_en_lemmas):
    synsets = wn.synsets(lemma, pos=wn_pos)
    if not synsets:
        continue

    senses = []
    for i, syn in enumerate(synsets, start=1):
        senses.append({
            "id": f"{lemma}.{POS_STR[wn_pos]}.{i:02d}",
            "gloss_en": syn.definition()
        })

    en_sense_inventory[lemma] = {
        "pos": {
            wn.NOUN: "NOUN",
            wn.VERB: "VERB",
            wn.ADJ: "ADJ",
            wn.ADV: "ADV"
        }[wn_pos],
        "senses": senses
    }

100%|██████████| 1869/1869 [00:00<00:00, 55713.80it/s]


In [ ]:
en_sense_inventory.get("money")

{'pos': 'NOUN',
 'senses': [{'id': 'money.n.01',
   'gloss_en': 'the most common medium of exchange; functions as legal tender'},
  {'id': 'money.n.02', 'gloss_en': 'wealth reckoned in terms of money'},
  {'id': 'money.n.03',
   'gloss_en': 'the official currency issued by a government or national bank'}]}

In [ ]:
with open("en_sense_list.json", "w", encoding="utf-8") as f:
    json.dump(en_sense_inventory, f, ensure_ascii=False, indent=2)

In [ ]:
import json
from collections import defaultdict

# =============================
# CONFIG
# =============================
INPUT_FILE = "final_en_sense_list.json"
OUTPUT_FILE = "final_en_sense_list_restructured.json"
MAX_SENSE_PER_POS = 5

# =============================
# LOAD DATA
# =============================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# =============================
# RESTRUCTURE
# =============================
# Output format:
# {
#   lemma: {
#       POS: [sense, sense, ...]
#   }
# }

restructured = defaultdict(lambda: defaultdict(list))

for lemma, entry in data.items():
    senses = entry.get("senses", [])

    for sense in senses:
        sense_id = sense.get("id")
        if not sense_id:
            continue

        # id format: lemma.pos.number  (e.g., pack.n.01)
        try:
            _, pos, _ = sense_id.split(".", 2)
            pos = pos.upper()
        except ValueError:
            continue  # skip malformed id

        restructured[lemma][pos].append(sense)

# =============================
# LIMIT SENSES PER POS
# =============================
final_data = {}

for lemma, pos_block in restructured.items():
    final_data[lemma] = {}
    for pos, senses in pos_block.items():
        final_data[lemma][pos] = senses[:MAX_SENSE_PER_POS]

# =============================
# SAVE OUTPUT
# =============================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=2)

print("✅ Sense list restructured successfully")
print(f"📄 Output file: {OUTPUT_FILE}")
print(f"📊 Total lemmas processed: {len(final_data)}")

✅ Sense list restructured successfully
📄 Output file: final_en_sense_list_restructured.json
📊 Total lemmas processed: 1128


In [ ]:
import json

# =============================
# CONFIG
# =============================
INPUT_FILE = "final_en_sense_list_restructured.json"
OUTPUT_FILE = "final_en_sense_list_polysemy_only.json"
REMOVED_SENSES_FILE = "removed_monosemic_senses.json"

# =============================
# LOAD DATA
# =============================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# =============================
# FILTER + LOG
# =============================
filtered_data = {}
removed_senses = []   # log toàn bộ sense bị xóa
removed_lemmas = []   # chỉ để thống kê

for lemma, pos_block in data.items():
    # Tổng số sense của lemma (gộp mọi POS)
    total_senses = sum(len(senses) for senses in pos_block.values())

    if total_senses > 1:
        filtered_data[lemma] = pos_block
    else:
        removed_lemmas.append(lemma)
        for pos, senses in pos_block.items():
            for sense in senses:
                removed_senses.append({
                    "lemma": lemma,
                    "pos": pos,
                    "id": sense.get("id"),
                    "gloss_en": sense.get("gloss_en", "")
                })

# =============================
# SAVE OUTPUT FILES
# =============================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

with open(REMOVED_SENSES_FILE, "w", encoding="utf-8") as f:
    json.dump(removed_senses, f, ensure_ascii=False, indent=2)

# =============================
# REPORT
# =============================
print("✅ Monosemic filtering completed")
print(f"📄 Remaining polysemous lemmas : {len(filtered_data)}")
print(f"🗑️ Removed monosemic lemmas   : {len(removed_lemmas)}")
print(f"📉 Removed senses logged      : {len(removed_senses)}")
print(f"📁 Sense log saved to         : {REMOVED_SENSES_FILE}")

✅ Monosemic filtering completed
📄 Remaining polysemous lemmas : 1085
🗑️ Removed monosemic lemmas   : 43
📉 Removed senses logged      : 43
📁 Sense log saved to         : removed_monosemic_senses.json


In [ ]:
# @title
import json
import nltk
from nltk.corpus import wordnet as wn

# =============================
# DOWNLOAD WORDNET (if needed)
# =============================
try:
    wn.synsets("test")
except LookupError:
    nltk.download("wordnet")
    nltk.download("omw-1.4")

# =============================
# CONFIG
# =============================
INPUT_FILE = "final_en_sense_list_polysemy_only.json"
OUTPUT_FILE = "wordnet_pos_suggestions.json"

# =============================
# LOAD DATA
# =============================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# =============================
# POS SUGGESTION (MATCH SENSE LIST)
# =============================
suggestions = {}

for lemma, pos_block in data.items():
    # POS hiện có trong sense list (n / v / a / r)
    current_pos = set(pos_block.keys())
    wn_pos_found = set()

    # Query WordNet
    for syn in wn.synsets(lemma):
        pos = syn.pos()
        if pos == "s":   # adjective satellite → adjective
            pos = "a"
        wn_pos_found.add(pos)

    missing_pos = wn_pos_found - current_pos

    if missing_pos:
        suggestions[lemma] = {
            "current_pos": sorted(list(current_pos)),
            "suggested_pos": sorted(list(missing_pos))
        }

# =============================
# SAVE OUTPUT
# =============================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(suggestions, f, ensure_ascii=False, indent=2)

# =============================
# REPORT
# =============================
print("✅ POS suggestion completed (POS normalized to WordNet codes)")
print(f"📊 Lemmas with missing POS: {len(suggestions)}")
print(f"📄 Suggestion file saved to: {OUTPUT_FILE}")

✅ POS suggestion completed (POS normalized to WordNet codes)
📊 Lemmas with missing POS: 1085
📄 Suggestion file saved to: wordnet_pos_suggestions.json


In [ ]:
import json
import nltk
from nltk.corpus import wordnet as wn

# =============================
# DOWNLOAD WORDNET (if needed)
# =============================
try:
    wn.synsets("test")
except LookupError:
    nltk.download("wordnet")
    nltk.download("omw-1.4")

# =============================
# CONFIG
# =============================
SENSE_LIST_FILE = "final_en_sense_list_polysemy_only.json"
SUGGESTION_FILE = "wordnet_pos_suggestions.json"

OUTPUT_FILE = "final_en_sense_list_extended.json"
ADDED_SENSES_FILE = "added_wordnet_senses.json"

MAX_SENSE_PER_POS = 5

# POS normalization: suggestion → WordNet code
POS_NORMALIZE = {
    "N": "n",
    "V": "v",
    "A": "a",
    "R": "r",
    "n": "n",
    "v": "v",
    "a": "a",
    "r": "r"
}

# POS for WordNet query
WN_QUERY_POS = {
    "n": wn.NOUN,
    "v": wn.VERB,
    "a": wn.ADJ,
    "r": wn.ADV
}

# =============================
# LOAD DATA
# =============================
with open(SENSE_LIST_FILE, "r", encoding="utf-8") as f:
    sense_list = json.load(f)

with open(SUGGESTION_FILE, "r", encoding="utf-8") as f:
    suggestions = json.load(f)

# =============================
# EXTEND SENSE LIST
# =============================
extended_sense_list = json.loads(json.dumps(sense_list))  # deep copy
added_senses_log = []

for lemma, info in suggestions.items():
    if lemma not in extended_sense_list:
        continue

    # ---- normalize POS ----
    current_pos = {
        POS_NORMALIZE[p]
        for p in info.get("current_pos", [])
        if p in POS_NORMALIZE
    }

    suggested_pos = {
        POS_NORMALIZE[p]
        for p in info.get("suggested_pos", [])
        if p in POS_NORMALIZE
    }

    # 👉 chỉ lấy POS còn thiếu
    missing_pos = suggested_pos - current_pos

    for pos in missing_pos:
        if pos not in WN_QUERY_POS:
            continue

        synsets = wn.synsets(lemma, pos=WN_QUERY_POS[pos])[:MAX_SENSE_PER_POS]
        if not synsets:
            continue

        extended_sense_list[lemma].setdefault(pos, [])

        for idx, syn in enumerate(synsets, start=1):
            sense_id = f"{lemma}.{pos}.{idx:02d}"

            sense_obj = {
                "id": sense_id,
                "gloss_en": syn.definition(),
                "source": "wordnet"
            }

            extended_sense_list[lemma][pos].append(sense_obj)

            added_senses_log.append({
                "lemma": lemma,
                "pos": pos,
                "id": sense_id,
                "gloss_en": syn.definition()
            })

# =============================
# SAVE OUTPUT FILES
# =============================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(extended_sense_list, f, ensure_ascii=False, indent=2)

with open(ADDED_SENSES_FILE, "w", encoding="utf-8") as f:
    json.dump(added_senses_log, f, ensure_ascii=False, indent=2)

# =============================
# REPORT
# =============================
print("✅ Sense list extended successfully (POS normalized)")
print(f"📄 Extended sense list : {OUTPUT_FILE}")
print(f"➕ Added senses        : {len(added_senses_log)}")
print(f"📁 Added sense log     : {ADDED_SENSES_FILE}")

✅ Sense list extended successfully (POS normalized)
📄 Extended sense list : final_en_sense_list_extended.json
➕ Added senses        : 2601
📁 Added sense log     : added_wordnet_senses.json


In [ ]:
!pip install -q transformers sentencepiece accelerate

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import re

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

SRC_LANG = "eng_Latn"
TGT_LANG = "vie_Latn"

In [ ]:
def translate_gloss_en_vi(text: str) -> str:
    tokenizer.src_lang = SRC_LANG

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_length=128,
            num_beams=4
        )

    return tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )

In [ ]:
INPUT_FILE = "en_sense_list.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    en_sense_data = json.load(f)

In [ ]:
vi_sense_data = {}

for lemma, pos_dict in tqdm(en_sense_data.items(), desc="Processing lemmas"):
    vi_sense_data[lemma] = {}

    for pos, senses in pos_dict.items():
        vi_sense_data[lemma][pos] = []

        for sense in senses:
            gloss_en = sense.get("gloss_en", "").strip()
            gloss_vi = translate_gloss_en_vi(gloss_en) if gloss_en else ""

            new_sense = sense.copy()
            new_sense["gloss_vi"] = gloss_vi

            vi_sense_data[lemma][pos].append(new_sense)

Processing lemmas: 100%|██████████| 1085/1085 [29:13<00:00,  1.62s/it]


In [ ]:
OUTPUT_FILE = "vi_sense_list.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(vi_sense_data, f, ensure_ascii=False, indent=2)

print(f"✅ Saved translated sense list to {OUTPUT_FILE}")

✅ Saved translated sense list to vi_sense_list.json
